In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-11-04 10:12:27.848703: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762251147.860197   11510 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762251147.863693   11510 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-04 10:12:27.876581: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1762251149.203689   11510 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18447 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-5:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-6e-6))

In [3]:
df_full = pd.read_hdf('../grids/Chiara-dnufit.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnufit'] = np.log10(df_full['dnufit'])

df_full['logAge'] = np.log10(df_full['age'])

df_full['logTeff'] = np.log10(df_full['Teff'])

df_full['logmassini'] = np.log10(df_full['massini'])

df_full['logyini'] = np.log10(df_full['yini'])

df_full['logalphaMLT'] = np.log10(df_full['alphaMLT'])


#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'logAge', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['lognumax', 'logdnufit'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [4]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'numax':0.001/3090, 'dnuSer':0.0001/135}
unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'lognumax':0.0001, 'logdnufit':0.00001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [23]:
n_dense_layers = 6

dense_layer_units = 128

Nepochs = 5000

learning_rate = 0.0001

model_name = 'logAge-logLPhot-lognumax-logdnuSer-exponent-6e-6-Adam'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [24]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='elu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='elu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/100000
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 524577.1875
Epoch 1: val_loss improved from inf to 508787.90625, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logall-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-100000-lrate-5e-05-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 524553.2500 - val_loss: 508787.9062 - learning_rate: 5.0000e-05
Epoch 2/100000
408/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 508805.9688
Epoch 2: val_loss improved from 508787.90625 to 504538.78125, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logall-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-100000-lrate-5e-05-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 508788.5938 - val_loss: 504538.7812 - learning_rate: 4.9999e-05
Epoch 3/100000
409/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 504088.6250
Epoch 3: val_loss improved from 504538.78125 to 497168.71875, saving model to /home/hatte/M4/models/long-ru

In [14]:
custom_objects

{'WMSE': <function WMSE.WMSE_metric(y_true, y_pred)>}

In [25]:
custom_objects =  {'WMSE':WMSE_metric}

model= tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)


In [26]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

model_name = model_name + '-50000'

print(model_name)

logAge-logLPhot-lognumax-logdnuSer-exponent-6e-6-Adam-50000


In [ ]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

from tensorflow.keras.mixed_precision import LossScaleOptimizer, global_policy

tf.keras.backend.clear_session()

adam = tf.keras.optimizers.Adam(learning_rate = 1e-6)

model.compile(loss=WMSE(weights), optimizer=adam)

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**12, #change higher
          verbose=1,
          epochs=450000,
          shuffle=True, callbacks = [tb_callback, cp_callback], initial_epoch = 400000) 


Epoch 400001/450000
1649/1649 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 8594.2412
Epoch 400001: val_loss improved from inf to 6003.28223, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logAge-logLPhot-lognumax-logdnuSer-exponent-6e-6-Adam-50000-nlayers-6-nunits-128-epochs-5000-lrate-0.0001-lossfunc-WMSE.model.keras
1649/1649 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 8593.1465 - val_loss: 6003.2822
Epoch 400002/450000
1645/1649 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5767.4805
Epoch 400002: val_loss improved from 6003.28223 to 5692.58984, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logAge-logLPhot-lognumax-logdnuSer-exponent-6e-6-Adam-50000-nlayers-6-nunits-128-epochs-5000-lrate-0.0001-lossfunc-WMSE.model.keras
1649/1649 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 5767.2710 - val_loss: 5692.5898
Epoch 400003/450000
1633/1649 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5570.1685
Epoch 400003: val_loss improved from 5692.58984 to 5615.77734, saving model to /ho

In [ ]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

nepochs = 5000 +Nepochs

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=Nepochs)